# 03 · Steroid receptor regulons (CollecTRI) across niches and cell types

**Question.** Do the AR / NR3C1 / PGR regulons change with age in the Xenium sections, and where in
the tissue (morphology niche) and in which cell types?

**Method in one paragraph.** Each pseudobulk from notebook 02 (donor × niche on patches, on
segmented cells, on fibroblast-compartment cells; donor × cell type on segmented cells) is
normalised to 10,000 counts and log1p-transformed on the 4,993-gene shared panel. Regulon activity
is the univariate linear model (ULM) t-statistic of `decoupler` for the **full CollecTRI** network
(`tmin = 5`, ~660 TFs measurable on the panel). AR, NR3C1 and PGR are read from that table; ESR1 /
ESR2 are kept in the tables for completeness. Because the full network is scored, each receptor can
also be expressed as its **percentile among all TFs within the same sample** — a shared ruler that
puts the three receptors on one axis.

**What can be claimed.** The absolute ULM level of a regulon on a 5 k marker panel is set by how
abundant its targets happen to be on the panel (the same NR3C1 regulon scores positive on the whole
transcriptome and negative here), so it is a per-source constant. Only **within-source**
comparisons are meaningful: age changes, and differences between niches or between cell types.
Every line plot below is therefore centred per source.

**Power.** One section per donor: 4 young (epaed U51, U48; lpaed Z16, Z17) vs 3 adult (A67, A72,
A70). The exact two-sided Mann–Whitney p cannot fall below 2/35 = 0.057, so stars (drawn from the
asymptotic Kruskal–Wallis p, BH-adjusted within source) are indicative only. Read the effect size
`delta_adult_vs_young` and `effect_r` (+1 = every adult section above every young section).

In [1]:
import os
import sys
import glob
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import decoupler as dc
from scipy.stats import kruskal, mannwhitneyu, spearmanr
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
sc.settings.verbosity = 0
import logging
logging.getLogger('fontTools').setLevel(logging.WARNING)      # PDF font subsetting otherwise logs every glyph
today = datetime.now().strftime('%Y-%m-%d')

REPO = '/nfs/team205/vk8/projects/thymus_ageing_atlas'
SPATIAL = f'{REPO}/Spatial_analyses'
GENERAL = f'{REPO}/General_analysis'
sys.path.insert(1, f'{GENERAL}/scripts')            # only for anno_levels (cell-type display order)
font_manager.fontManager.addfont('/nfs/team205/ny1/ThymusSpatialAtlas/software/Arial.ttf')
plt.style.use(f'{GENERAL}/scripts/plotting/thyAgeing.mplstyle')
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)

PB_DIR = f'{SPATIAL}/data/xenium/pseudobulk_final'
RESULTS = f'{SPATIAL}/results/xenium/final/03_steroid_regulons'
PLOTS = f'{SPATIAL}/plots/xenium/final/03_steroid_regulons'
for d in (RESULTS, PLOTS):
    os.makedirs(d, exist_ok=True)
PREFIX = 'xenium_steroid'

DONOR_AGE = {'U51': 'epaed', 'U48': 'epaed', 'Z16': 'lpaed', 'Z17': 'lpaed', 'A67': 'adult', 'A72': 'adult', 'A70': 'adult'}
DONOR_ORDER = ['U51', 'U48', 'Z16', 'Z17', 'A67', 'A72', 'A70']
AGE_ORDER = ['epaed', 'lpaed', 'adult']
MORPHO_L2_ORDER = ['cortex_septal', 'cortex', 'medulla', 'medulla_hc', 'adipose', 'fibrosis_adipose', 'fibrosis', 'vessel']
MORPHO_L3_ORDER = ['cortex_septal', 'cortex_outer', 'cortex_dense', 'cortex', 'cortex_residual',
                   'medulla', 'medulla_hc', 'medulla_residual', 'adipose', 'adipose_edge',
                   'fibrosis', 'fibrosis_adipose', 'endothelium_vessel', 'endothelium_edge']
MORPHO_ORDERS = {'morpho_niche_l2': MORPHO_L2_ORDER, 'morpho_niche_l3': MORPHO_L3_ORDER}

RECEPTORS = ['AR', 'NR3C1', 'PGR', 'ESR1', 'ESR2']
FOCUS = ['AR', 'NR3C1', 'PGR']
TMIN = 5
print(f'{today} | decoupler {dc.__version__} | scanpy {sc.__version__}')

2026-09-04 | decoupler 2.1.1 | scanpy 1.11.1


## 1 · Pseudobulks

Eight units, all on the shared panel. `representation` × `level`:
patches / cells / fbcells × morphology niche (l2, l3), and cells × cell type (`taa_l2_upd`, `taa_l3_upd`).

In [2]:
UNITS = [('patches', 'morpho_niche_l2'), ('patches', 'morpho_niche_l3'),
         ('cells', 'morpho_niche_l2'), ('cells', 'morpho_niche_l3'),
         ('cells', 'taa_l2_upd'), ('cells', 'taa_l3_upd'),
         ('fbcells', 'morpho_niche_l2'), ('fbcells', 'morpho_niche_l3')]
META = ['donor', 'group', 'level', 'representation', 'age_group', 'age_group2', 'age_years', 'psbulk_cells', 'psbulk_counts_shared']


def load_pb(rep, level):
    a = ad.read_h5ad(f'{PB_DIR}/pb_{rep}_{level}.h5ad')
    a = a[:, a.var['shared_panel'].to_numpy()].copy()
    a.obs['age_group'] = pd.Categorical(a.obs['age_group'].astype(str), categories=AGE_ORDER)
    a.obs['age_group2'] = pd.Categorical(a.obs['age_group2'].astype(str), categories=['young', 'adult'])
    return a


PBS = {u: load_pb(*u) for u in UNITS}
PANEL = PBS[('cells', 'morpho_niche_l2')].var_names
assert all(list(a.var_names) == list(PANEL) for a in PBS.values()), 'pseudobulks are not on one gene universe'
for (rep, lv), a in PBS.items():
    print(f'{rep:8s} {lv:16s} {a.n_obs:4d} samples | {a.obs["group"].nunique():3d} groups | {a.obs["donor"].nunique()} donors')
print(f'\nshared panel: {len(PANEL):,} genes')


def ct_order(level, present):
    """Cell types in the curated display order; anything unknown to the curated table goes last."""
    try:
        from anno_levels import get_ct_levels
        base = get_ct_levels({'taa_l2_upd': 'taa_l2_v2', 'taa_l3_upd': 'taa_l3'}[level])
    except Exception as e:                                          # noqa: BLE001
        print(f'anno_levels unavailable ({e}); alphabetical order')
        base = []
    return [c for c in base if c in present] + sorted(c for c in present if c not in base)


ORDERS = dict(MORPHO_ORDERS)
for lv in ['taa_l2_upd', 'taa_l3_upd']:
    ORDERS[lv] = ct_order(lv, set(PBS[('cells', lv)].obs['group']))

patches  morpho_niche_l2    56 samples |   8 groups | 7 donors
patches  morpho_niche_l3    82 samples |  13 groups | 7 donors
cells    morpho_niche_l2    56 samples |   8 groups | 7 donors
cells    morpho_niche_l3    96 samples |  14 groups | 7 donors
cells    taa_l2_upd        188 samples |  29 groups | 7 donors
cells    taa_l3_upd        316 samples |  51 groups | 7 donors
fbcells  morpho_niche_l2    46 samples |   8 groups | 7 donors
fbcells  morpho_niche_l3    68 samples |  13 groups | 7 donors

shared panel: 4,993 genes


Latest file: thyAgeing_full_curatedAnno_v12_2025-09-02_levels.xlsx


Latest file: thyAgeing_full_curatedAnno_v12_2025-09-02_levels.xlsx


## 2 · Network and panel coverage

How much of each receptor's regulon the panel measures, and its sign balance (a mostly-positive
target set is what makes the raw level track target abundance).

In [3]:
collectri = dc.op.collectri(organism='human')[['source', 'target', 'weight']]
rows = []
for s in RECEPTORS:
    d = collectri[collectri['source'] == s]
    d_in = d[d['target'].isin(PANEL)]
    rows.append({'source': s, 'n_targets': len(d), 'n_in_panel': len(d_in), 'frac_in_panel': len(d_in) / len(d),
                 'n_pos_in_panel': int((d_in['weight'] > 0).sum()), 'n_neg_in_panel': int((d_in['weight'] < 0).sum()),
                 'receptor_on_panel': s in PANEL})
coverage = pd.DataFrame(rows).set_index('source')
coverage.to_csv(f'{RESULTS}/{PREFIX}_panelCoverage.csv')
print(f'CollecTRI: {collectri["source"].nunique():,} TFs, {len(collectri):,} edges')
coverage.round(3)

CollecTRI: 1,185 TFs, 42,990 edges


,n_targets,n_in_panel,frac_in_panel,n_pos_in_panel,n_neg_in_panel,receptor_on_panel
source,,,,,,
AR,514,302,0.588,270,32,True
NR3C1,292,191,0.654,144,47,True
PGR,133,88,0.662,71,17,True
ESR1,634,379,0.598,349,30,True
ESR2,208,127,0.611,109,18,True


## 3 · Scoring

`sc.pp.normalize_total(1e4)` → `log1p` → `dc.mt.ulm` with the full CollecTRI net. Two tables come
out: the receptors' ULM t (`table1_scores`) and their percentile among all measurable TFs within
each sample (`table1_percentiles`).

In [4]:
def score_ulm(pb, net, tmin=TMIN):
    p = pb.copy()
    sc.pp.normalize_total(p, target_sum=1e4)
    sc.pp.log1p(p)
    dc.mt.ulm(data=p, net=net, tmin=tmin, verbose=False)
    scores = p.obsm.pop('score_ulm')
    p.obsm.pop('padj_ulm', None)
    return p, scores


def long_table(scores, obs, sources, value='score'):
    m = scores[sources].stack().rename(value).rename_axis(['sample', 'source']).reset_index()
    return m.join(obs[META], on='sample')


frames, pct_frames, NORMED, n_tf = [], [], {}, {}
for (rep, lv), a in PBS.items():
    p, scores = score_ulm(a, collectri)
    assert all(r in scores.columns for r in RECEPTORS), f'{rep}/{lv}: receptor pruned by tmin'
    NORMED[(rep, lv)] = p
    n_tf[(rep, lv)] = scores.shape[1]
    frames.append(long_table(scores, p.obs, RECEPTORS))
    pct_frames.append(long_table(scores.rank(axis=1, pct=True) * 100.0, p.obs, RECEPTORS, value='score'))
assert len(set(n_tf.values())) == 1, f'TF reference set differs between units: {n_tf}'
t1 = pd.concat(frames, ignore_index=True)
t1pct = pd.concat(pct_frames, ignore_index=True)
for t in (t1, t1pct):
    t['age_group'] = pd.Categorical(t['age_group'].astype(str), categories=AGE_ORDER)
t1.to_csv(f'{RESULTS}/{PREFIX}_table1_scores.csv', index=False)
t1pct.to_csv(f'{RESULTS}/{PREFIX}_table1_percentiles.csv', index=False)
print(f'{len(t1):,} sample x receptor rows; reference set {list(n_tf.values())[0]} TFs (tmin={TMIN}) in every unit')
print('\nmean ULM t per receptor and age group, segmented cells x morpho_niche_l2 (mean over niches):')
(t1[(t1['representation'] == 'cells') & (t1['level'] == 'morpho_niche_l2')]
 .pivot_table(index='source', columns='age_group', values='score', observed=True).reindex(RECEPTORS).round(2))

4,540 sample x receptor rows; reference set 663 TFs (tmin=5) in every unit

mean ULM t per receptor and age group, segmented cells x morpho_niche_l2 (mean over niches):


age_group,epaed,lpaed,adult
source,,,
AR,1.55,1.45,2.76
NR3C1,-4.16,-4.28,-2.36
PGR,-0.51,-0.61,0.51
ESR1,-0.61,-0.82,0.06
ESR2,-1.03,-0.97,-0.23


## 4 · Age: adult (n = 3) vs young = epaed + lpaed (n = 4), per niche and per cell type

In [5]:
def young_vs_adult(t, gkeys, value='score'):
    rows = []
    for keys, d in t.groupby(gkeys, observed=True):
        y = d.loc[d['age_group2'] == 'young', value].to_numpy(float)
        a = d.loc[d['age_group2'] == 'adult', value].to_numpy(float)
        if len(y) < 2 or len(a) < 2:
            continue
        try:
            p_kw = float(kruskal(y, a)[1])
        except ValueError:
            p_kw = np.nan
        U = float(mannwhitneyu(a, y, alternative='two-sided')[0])
        try:
            p_exact = float(mannwhitneyu(a, y, alternative='two-sided', method='exact')[1])
        except Exception:                                           # noqa: BLE001  (ties)
            p_exact = np.nan
        means = d.groupby('age_group', observed=True)[value].mean()
        rows.append({**dict(zip(gkeys, keys)), 'n_young': len(y), 'n_adult': len(a),
                     'mean_young': y.mean(), 'mean_adult': a.mean(), 'delta_adult_vs_young': a.mean() - y.mean(),
                     'effect_r': 2 * U / (len(a) * len(y)) - 1, 'p_kw': p_kw, 'p_exact': p_exact,
                     **{f'mean_{ag}': float(means.get(ag, np.nan)) for ag in AGE_ORDER}})
    t2 = pd.DataFrame(rows)
    t2['padj_kw'] = t2.groupby(['representation', 'level', 'source'], observed=True)['p_kw'].transform(
        lambda p: multipletests(p.fillna(1), method='fdr_bh')[1])
    return t2


GKEYS = ['representation', 'level', 'group', 'source']
t2 = young_vs_adult(t1, GKEYS)
t2.to_csv(f'{RESULTS}/{PREFIX}_table2_ageStats.csv', index=False)
t2pct = young_vs_adult(t1pct, GKEYS)
t2pct.to_csv(f'{RESULTS}/{PREFIX}_table2_ageStats_percentiles.csv', index=False)
print(f'{len(t2):,} contrasts | exact p floor {2 / 35:.4f}, observed min {t2["p_exact"].min():.4f} | '
      f'asymptotic padj<0.05: {int((t2["padj_kw"] < 0.05).sum())}; complete separation (|effect_r| = 1): {int((t2["effect_r"].abs() == 1).sum())}')

cols = ['source', 'group', 'mean_young', 'mean_adult', 'delta_adult_vs_young', 'effect_r', 'padj_kw', 'p_exact']
print('\nNICHE level (segmented cells, morpho_niche_l2):')
print(t2[(t2['representation'] == 'cells') & (t2['level'] == 'morpho_niche_l2') & t2['source'].isin(FOCUS)]
      .pivot_table(index='group', columns='source', values='delta_adult_vs_young').reindex(MORPHO_L2_ORDER)[FOCUS].round(2).to_string())
print('\nNICHE level (transcript patches, morpho_niche_l2):')
print(t2[(t2['representation'] == 'patches') & (t2['level'] == 'morpho_niche_l2') & t2['source'].isin(FOCUS)]
      .pivot_table(index='group', columns='source', values='delta_adult_vs_young').reindex(MORPHO_L2_ORDER)[FOCUS].round(2).to_string())
print('\nCELL-TYPE level (taa_l2_upd), largest adult - young changes:')
print(t2[(t2['representation'] == 'cells') & (t2['level'] == 'taa_l2_upd') & t2['source'].isin(FOCUS)]
      .sort_values('delta_adult_vs_young', ascending=False)[cols].head(15).round(3).to_string(index=False))

635 contrasts | exact p floor 0.0571, observed min 0.0571 | asymptotic padj<0.05: 119; complete separation (|effect_r| = 1): 395

NICHE level (segmented cells, morpho_niche_l2):
source              AR  NR3C1   PGR
group                              
cortex_septal     1.22   1.92  1.15
cortex            1.13   1.60  0.97
medulla           1.07   1.64  0.75
medulla_hc        1.00   1.38  0.68
adipose           1.49   2.14  1.28
fibrosis_adipose  1.43   2.12  1.29
fibrosis          1.44   2.16  1.24
vessel            1.25   1.89  1.17

NICHE level (transcript patches, morpho_niche_l2):
source              AR  NR3C1   PGR
group                              
cortex_septal     1.27   1.95  1.18
cortex            1.23   1.68  0.99
medulla           1.12   1.69  0.79
medulla_hc        1.04   1.41  0.72
adipose           1.62   2.25  1.30
fibrosis_adipose  1.59   2.27  1.38
fibrosis          1.53   2.23  1.30
vessel            1.31   1.92  1.14

CELL-TYPE level (taa_l2_upd), largest adult - you

## 5 · Figures — heatmaps per age group, and the adult − young change

Nature sizing: 89 mm (8 niches), 130 mm (14 niches) or 183 mm (cell types) wide; text sizes from
`thyAgeing.mplstyle`; saved at the literal canvas size (`bbox_inches=None`), so the PDFs tile.

In [6]:
NATURE_W = {'single': 89.0, 'one-half': 130.0, 'double': 183.0}
MAX_H_MM = 247.0
FIG_WIDTH = {'morpho_niche_l2': 'single', 'morpho_niche_l3': 'one-half', 'taa_l2_upd': 'double', 'taa_l3_upd': 'double'}
FIG_SIZES = []
AGE_COLORS = {'epaed': '#9ecae1', 'lpaed': '#4292c6', 'adult': '#08306b'}
SRC_COLORS = {'NR3C1': '#DC267F', 'AR': '#648FFF', 'PGR': '#FE6100', 'ESR1': '#785EF0', 'ESR2': '#FFB000'}
LEG = dict(frameon=False, handlelength=1.2, labelspacing=0.25, borderpad=0.2)


def nature_size(n_rows, groups, width='double', row_mm=4.2, pad_mm=16.0):
    w_mm = NATURE_W.get(width, width)
    xlab_mm = 1.05 * max(len(str(g)) for g in groups) + 2.0           # rotated x labels cost height
    return (w_mm / 25.4, min(MAX_H_MM, n_rows * row_mm + xlab_mm + pad_mm) / 25.4)


def save_exact(fig, path):
    with plt.rc_context({'savefig.bbox': 'standard'}):      # the house style sets 'tight', which re-crops the canvas
        fig.savefig(path)
    w, h = np.asarray(fig.get_size_inches()) * 25.4
    FIG_SIZES.append({'figure': os.path.basename(path), 'width_mm': round(float(w), 1), 'height_mm': round(float(h), 1)})


def stars(p):
    return '' if not np.isfinite(p) else ('***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '')


def plot_age_blocks(t1, t2, rep, level, sources, title, save=None, value_label='ULM t'):
    """Sources x groups: one heatmap per age group, then the adult - young change with stars."""
    d1 = t1[(t1['representation'] == rep) & (t1['level'] == level) & t1['source'].isin(sources)]
    d2 = t2[(t2['representation'] == rep) & (t2['level'] == level) & t2['source'].isin(sources)]
    groups = [g for g in ORDERS[level] if g in set(d1['group'])]
    srcs = [s for s in sources if s in set(d1['source'])]
    blocks = [(ag, d1[d1['age_group'] == ag].pivot_table(index='source', columns='group', values='score')
               .reindex(index=srcs, columns=groups)) for ag in AGE_ORDER]
    delta = d2.pivot_table(index='source', columns='group', values='delta_adult_vs_young').reindex(index=srcs, columns=groups)
    padj = d2.pivot_table(index='source', columns='group', values='padj_kw').reindex(index=srcs, columns=groups)
    vmax_act = np.nanmax(np.abs(np.concatenate([b[1].to_numpy().ravel() for b in blocks])))
    vmax_del = np.nanmax(np.abs(delta.to_numpy()))
    fig, axes = plt.subplots(len(blocks) + 1, 1, sharex=True, layout='constrained',
                             figsize=nature_size(len(srcs) * (len(blocks) + 1), groups, FIG_WIDTH[level], pad_mm=22.0))
    for ax, (name, mat) in zip(axes, blocks + [('adult − young', delta)]):
        is_delta = name.startswith('adult −')
        sns.heatmap(mat, ax=ax, cmap='vlag', center=0, vmin=-(vmax_del if is_delta else vmax_act), vmax=(vmax_del if is_delta else vmax_act),
                    annot=padj.map(stars) if is_delta else None, fmt='', annot_kws={'fontsize': 5},
                    linewidths=0.3, linecolor='white', cbar_kws={'label': 'Δ' if is_delta else value_label, 'pad': 0.015, 'aspect': 12})
        ax.set_ylabel(name, rotation=0, ha='right', va='center')
        ax.set_xlabel('')
        ax.tick_params(axis='y', rotation=0)
    axes[-1].set_xticklabels(axes[-1].get_xticklabels(), rotation=90)
    fig.suptitle(title)
    if save:
        save_exact(fig, save)
    return fig


def plot_delta(t2, rep, level, sources, title, save=None, star_col='padj_kw'):
    """Sources x groups heatmap of adult - young; stars from the asymptotic KW padj, dot = complete separation."""
    d2 = t2[(t2['representation'] == rep) & (t2['level'] == level) & t2['source'].isin(sources)]
    groups = [g for g in ORDERS[level] if g in set(d2['group'])]
    srcs = [s for s in sources if s in set(d2['source'])]
    delta = d2.pivot_table(index='source', columns='group', values='delta_adult_vs_young').reindex(index=srcs, columns=groups)
    padj = d2.pivot_table(index='source', columns='group', values=star_col).reindex(index=srcs, columns=groups)
    er = d2.pivot_table(index='source', columns='group', values='effect_r').reindex(index=srcs, columns=groups)
    ann = padj.map(stars) + np.where(er.abs().to_numpy() == 1, '•', '')
    vmax = np.nanmax(np.abs(delta.to_numpy()))
    fig, ax = plt.subplots(layout='constrained', figsize=nature_size(len(srcs), groups, FIG_WIDTH[level], row_mm=5.0, pad_mm=14.0))
    sns.heatmap(delta, ax=ax, cmap='vlag', center=0, vmin=-vmax, vmax=vmax, annot=ann, fmt='', annot_kws={'fontsize': 5},
                linewidths=0.3, linecolor='white', cbar_kws={'label': 'Δ (adult − young)', 'pad': 0.015, 'aspect': 12})
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
    ax.tick_params(axis='y', rotation=0)
    ax.set_title(title)
    if save:
        save_exact(fig, save)
    return fig

In [7]:
for (rep, lv) in UNITS:
    plot_age_blocks(t1, t2, rep, lv, FOCUS, f'AR / NR3C1 / PGR — {rep}, {lv}', save=f'{PLOTS}/{PREFIX}_ageBlocks_{rep}_{lv}.pdf')
    plot_delta(t2, rep, lv, FOCUS, f'adult − young — {rep}, {lv}', save=f'{PLOTS}/{PREFIX}_delta_{rep}_{lv}.pdf')
    if rep == 'cells' and lv == 'morpho_niche_l2':
        plt.show()
    plt.close('all')
# all five receptors, cell-type level, for the supplement
for lv in ['taa_l2_upd', 'taa_l3_upd']:
    plot_age_blocks(t1, t2, 'cells', lv, RECEPTORS, f'five receptors — cells, {lv}', save=f'{PLOTS}/{PREFIX}_ageBlocks_NR5_cells_{lv}.pdf')
    plt.close('all')
print('figures ->', PLOTS)

figures -> /nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/plots/xenium/final/03_steroid_regulons


## 6 · Normalised line plots across niches

Three views, all on the three receptors, niche order along the involution axis:

- **profile** — one receptor, one panel per representation (patches | cells | fibroblast cells),
  one line per age group, donors as faint points;
- **overlay** — the three receptors on one axis, one panel per age group. Each receptor is centred
  on its own mean over the niche × age grid (`centre='grid'`, the age shift stays visible) or on its
  mean within each age group (`centre='age'`, only the niche shape survives — needed because the age
  shift is several times the niche gradient);
- **percentile** — the receptor's rank among all measurable TFs within the sample: a shared ruler,
  no centring needed, but it saturates near the top.

In [8]:
def niche_pivot(tab, rep, level, src, centre=False):
    d = tab[(tab['representation'] == rep) & (tab['level'] == level) & (tab['source'] == src)]
    order = [g for g in MORPHO_ORDERS[level] if g in set(d['group'])]
    piv = d.pivot_table(index='group', columns='age_group', values='score', observed=True).reindex(index=order, columns=AGE_ORDER)
    if centre == 'age':
        piv = piv - piv.mean(axis=0)
    elif centre:
        piv = piv - np.nanmean(piv.values)
    return piv, d


def line_size(n_rows, groups, width, ax_mm=26.0, pad_mm=13.0):
    h_mm = n_rows * ax_mm + 1.05 * max(len(str(g)) for g in groups) + 2.0 + pad_mm
    return (NATURE_W[width] / 25.4, min(MAX_H_MM, h_mm) / 25.4)


def niche_axis(ax, order):
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=90)
    ax.grid(axis='y', color='0.92', lw=0.5, linestyle='-')
    ax.set_axisbelow(True)
    ax.margins(x=0.04)


def plot_niche_profile(tab, src, level, reps=('patches', 'cells', 'fbcells'), ylab=None, save=None):
    order = [g for g in MORPHO_ORDERS[level] if g in set(tab.loc[tab['level'] == level, 'group'])]
    fig, axes = plt.subplots(1, len(reps), sharey=True, layout='constrained', figsize=line_size(1, order, 'one-half' if len(reps) < 3 else 'double', ax_mm=30.0))
    pos = {g: i for i, g in enumerate(order)}
    for ax, rep in zip(np.atleast_1d(axes), reps):
        piv, d = niche_pivot(tab, rep, level, src)
        piv = piv.reindex(order)
        for ag in AGE_ORDER:
            dd = d[(d['age_group'] == ag) & d['group'].isin(order)]
            ax.scatter(dd['group'].map(pos) + (AGE_ORDER.index(ag) - 1) * 0.16, dd['score'], s=2, lw=0, alpha=0.5, color=AGE_COLORS[ag], zorder=1)
            ax.plot(range(len(order)), piv[ag], marker='o', ms=2.5, lw=1.0, color=AGE_COLORS[ag], label=ag, zorder=3)
        ax.set_title({'patches': 'transcript patches', 'cells': 'segmented cells', 'fbcells': 'fibroblast cells'}[rep])
        niche_axis(ax, order)
    np.atleast_1d(axes)[0].set_ylabel(ylab or f'{src} ULM t')
    np.atleast_1d(axes)[0].legend(title='age group', **LEG)
    fig.suptitle(f'{src} — {level}')
    if save:
        save_exact(fig, save)
    return fig


def plot_niche_overlay(tab, level, rep, sources=FOCUS, centre='grid', ylab=None, save=None):
    order = [g for g in MORPHO_ORDERS[level] if g in set(tab.loc[(tab['level'] == level) & (tab['representation'] == rep), 'group'])]
    pivs = {s: niche_pivot(tab, rep, level, s, centre=centre)[0].reindex(order) for s in sources}
    fig, axes = plt.subplots(1, len(AGE_ORDER), sharey=True, sharex=True, layout='constrained', figsize=line_size(1, order, 'double', ax_mm=30.0, pad_mm=16.0))
    for ax, ag in zip(axes, AGE_ORDER):
        for s in sources:
            ax.plot(range(len(order)), pivs[s][ag], marker='o', ms=2.5, lw=1.0, color=SRC_COLORS[s], label=s)
        if centre:
            ax.axhline(0, color='0.6', lw=0.5, zorder=0)
        ax.set_title(ag)
        niche_axis(ax, order)
    axes[0].set_ylabel(ylab or ('ULM t, centred per receptor' + (' within age group' if centre == 'age' else '')))
    axes[0].legend(**LEG)
    fig.suptitle(f'{rep}, {level}' + (' — niche shape only' if centre == 'age' else ''))
    if save:
        save_exact(fig, save)
    return fig


for src in FOCUS:
    plot_niche_profile(t1, src, 'morpho_niche_l2', save=f'{PLOTS}/{PREFIX}_nicheProfile_{src}_morpho_niche_l2.pdf')
    plt.show()
    plot_niche_profile(t1, src, 'morpho_niche_l3', save=f'{PLOTS}/{PREFIX}_nicheProfile_{src}_morpho_niche_l3.pdf')
    plot_niche_profile(t1pct, src, 'morpho_niche_l2', ylab=f'{src} percentile among TFs', save=f'{PLOTS}/{PREFIX}_nichePercentile_{src}_morpho_niche_l2.pdf')
    plt.close('all')
for level in ['morpho_niche_l2', 'morpho_niche_l3']:
    for rep in ['patches', 'cells', 'fbcells']:
        plot_niche_overlay(t1, level, rep, centre='grid', save=f'{PLOTS}/{PREFIX}_nicheOverlay_{rep}_{level}.pdf')
        plot_niche_overlay(t1, level, rep, centre='age', save=f'{PLOTS}/{PREFIX}_nicheShape_{rep}_{level}.pdf')
        plot_niche_overlay(t1pct, level, rep, centre=False, ylab='percentile among CollecTRI TFs', save=f'{PLOTS}/{PREFIX}_nichePercentile_{rep}_{level}.pdf')
        if rep == 'cells' and level == 'morpho_niche_l2':
            plt.show()
        plt.close('all')

## 7 · Which niche scores highest, per age group — and do the receptors agree?

In [9]:
rank_rows = []
for (rep, lv) in [u for u in UNITS if u[1].startswith('morpho')]:
    g = (t1[(t1['representation'] == rep) & (t1['level'] == lv) & t1['source'].isin(FOCUS)]
         .groupby(['source', 'age_group', 'group'], observed=True)['score'].mean().reset_index())
    g['rank'] = g.groupby(['source', 'age_group'], observed=True)['score'].rank(ascending=False)
    g['representation'], g['level'] = rep, lv
    rank_rows.append(g)
niche_rank = pd.concat(rank_rows, ignore_index=True)
niche_rank.to_csv(f'{RESULTS}/{PREFIX}_nicheRanking.csv', index=False)
print('highest-scoring niche per receptor and age group (morpho_niche_l2):')
print(niche_rank[(niche_rank['level'] == 'morpho_niche_l2') & (niche_rank['rank'] == 1)]
      .pivot_table(index=['representation', 'source'], columns='age_group', values='group', aggfunc='first', observed=True).to_string())

conc = []
for (rep, lv) in [u for u in UNITS if u[1].startswith('morpho')]:
    c = pd.DataFrame({s: niche_pivot(t1, rep, lv, s, centre=True)[0].mean(axis=1) for s in FOCUS}).corr(method='spearman')
    for a_, b_ in [('NR3C1', 'AR'), ('NR3C1', 'PGR'), ('AR', 'PGR')]:
        conc.append({'representation': rep, 'level': lv, 'pair': f'{a_} vs {b_}', 'rho_niche_profile': round(float(c.loc[a_, b_]), 3)})
conc = pd.DataFrame(conc)
conc.to_csv(f'{RESULTS}/{PREFIX}_nicheProfileConcordance.csv', index=False)
print('\nSpearman between the receptors\' age-averaged niche profiles:')
conc.pivot_table(index=['level', 'representation'], columns='pair', values='rho_niche_profile')

highest-scoring niche per receptor and age group (morpho_niche_l2):
age_group                      epaed             lpaed             adult
representation source                                                   
cells          AR         medulla_hc        medulla_hc  fibrosis_adipose
               NR3C1      medulla_hc        medulla_hc  fibrosis_adipose
               PGR        medulla_hc        medulla_hc  fibrosis_adipose
fbcells        AR             cortex           adipose           adipose
               NR3C1   cortex_septal  fibrosis_adipose           adipose
               PGR           adipose            vessel            vessel
patches        AR         medulla_hc        medulla_hc  fibrosis_adipose
               NR3C1      medulla_hc        medulla_hc  fibrosis_adipose
               PGR            vessel        medulla_hc  fibrosis_adipose

Spearman between the receptors' age-averaged niche profiles:


pair                            AR vs PGR  NR3C1 vs AR  NR3C1 vs PGR
level           representation                                      
morpho_niche_l2 cells               0.548        0.952         0.476
                fbcells             0.262       -0.095         0.381
                patches             0.619        0.952         0.500
morpho_niche_l3 cells               0.877        0.895         0.692
                fbcells             0.368        0.275         0.231
                patches             0.654        0.775         0.269

## 8 · Cross-modality check against the scRNA-seq age contrast (cell-type level)

With seven sections, rank agreement with the scRNA-seq cohort's adult − infant change is the
best-powered statement available. Skipped if the scRNA-seq table is absent.

In [10]:
SC_T2 = f'{GENERAL}/results/scRNAseq/steroids_v4/thyAgeing_scRNAseq_steroid_agestatsRAW_taa_l2_v2.csv'
if os.path.exists(SC_T2):
    sc_t2 = pd.read_csv(SC_T2)
    xen = t2[(t2['representation'] == 'cells') & (t2['level'] == 'taa_l2_upd') & t2['source'].isin(RECEPTORS)]
    mrg = xen.merge(sc_t2[['source', 'cell_type', 'delta_adult_vs_infant']], left_on=['source', 'group'], right_on=['source', 'cell_type'])
    mrg = mrg.dropna(subset=['delta_adult_vs_young', 'delta_adult_vs_infant'])
    rho, pval = spearmanr(mrg['delta_adult_vs_young'], mrg['delta_adult_vs_infant'])
    mrg.to_csv(f'{RESULTS}/{PREFIX}_crossModality_taa_l2.csv', index=False)
    fig, ax = plt.subplots(layout='constrained', figsize=(NATURE_W['single'] / 25.4, 80 / 25.4))
    for src, d in mrg.groupby('source'):
        ax.scatter(d['delta_adult_vs_infant'], d['delta_adult_vs_young'], s=6, label=src, alpha=0.85, lw=0.3, color=SRC_COLORS.get(src))
    ax.axhline(0, color='0.6', lw=0.5)
    ax.axvline(0, color='0.6', lw=0.5)
    ax.set_xlabel('scRNA-seq  Δ adult − infant')
    ax.set_ylabel('Xenium  Δ adult − young')
    ax.set_title(f'ρ = {rho:.2f}, p = {pval:.1e}, n = {len(mrg)}')
    ax.legend(**LEG, ncol=2)
    save_exact(fig, f'{PLOTS}/{PREFIX}_crossModality_taa_l2.pdf')
    plt.show()
    print(f'Spearman rho = {rho:.3f} (p = {pval:.2g}) over {len(mrg)} receptor x cell-type pairs')
else:
    print('scRNA-seq table not found — cross-modality check skipped')

Spearman rho = 0.516 (p = 7.2e-10) over 125 receptor x cell-type pairs


In [11]:
sizes = pd.DataFrame(FIG_SIZES).drop_duplicates('figure').sort_values('figure')
sizes.to_csv(f'{RESULTS}/{PREFIX}_figureSizes.csv', index=False)
assert sizes['width_mm'].round(0).isin([89, 130, 183]).all() and (sizes['height_mm'] <= MAX_H_MM + 0.5).all()
print(f'{len(sizes)} figures, all at a Nature column width and within 247 mm')
print(f'tables  -> {RESULTS}')
print(f'figures -> {PLOTS}')
for f_ in sorted(os.listdir(RESULTS)):
    print('  ', f_)

46 figures, all at a Nature column width and within 247 mm
tables  -> /nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/results/xenium/final/03_steroid_regulons
figures -> /nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/plots/xenium/final/03_steroid_regulons
   xenium_steroid_crossModality_taa_l2.csv
   xenium_steroid_figureSizes.csv
   xenium_steroid_nicheProfileConcordance.csv
   xenium_steroid_nicheRanking.csv
   xenium_steroid_panelCoverage.csv
   xenium_steroid_table1_percentiles.csv
   xenium_steroid_table1_scores.csv
   xenium_steroid_table2_ageStats.csv
   xenium_steroid_table2_ageStats_percentiles.csv
